In [ ]:
!pip install gTTS

In [ ]:
!pip install gTTS pandas openpyxl scikit-learn matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import ipywidgets as widgets
from IPython.display import display, Audio, clear_output
from gtts import gTTS
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error


class PersonalFinanceApp:

    def __init__(self):
        self.file_name = "/content/drive/MyDrive/finance_data.xlsx"
        self.last_income = None
        self.last_expenses = None
        self.create_widgets()


    def speak(self, text):
        try:
            tts = gTTS(text=text, lang='en')
            tts.save("output.mp3")
            display(Audio("output.mp3", autoplay=True))
        except Exception:
            display(widgets.Label(value="Audio unavailable in this environment."))

    def fmt(self, amount):
        return f"{amount:,.2f}"

    def save_to_excel(self, income, expenses):
        new_data = pd.DataFrame([[income, expenses]], columns=["Income", "Expenses"])
        if os.path.exists(self.file_name):
            old_data = pd.read_excel(self.file_name)
            updated_data = pd.concat([old_data, new_data], ignore_index=True)
        else:
            updated_data = new_data
        updated_data.to_excel(self.file_name, index=False)

    def load_data(self):
        if os.path.exists(self.file_name):
            return pd.read_excel(self.file_name)
        return None

    def back_button(self):
        btn = widgets.Button(description="Back", button_style='danger')
        btn.on_click(lambda _: self.create_widgets())
        return btn



    def train_models(self):
        data = self.load_data()
        if data is None or len(data) < 5:
            return None, None, None

        X = data[["Income"]]
        y = data["Expenses"]

        linear_model = LinearRegression()
        linear_model.fit(X, y)

        poly = PolynomialFeatures(degree=2)
        X_poly = poly.fit_transform(X)
        poly_model = LinearRegression()
        poly_model.fit(X_poly, y)

        return linear_model, poly_model, poly



    def generate_ai_suggestions(self, income, expenses):
        suggestions = []
        savings = income - expenses

        if expenses > income:
            suggestions.append("WARNING: You are spending more than you earn. Cut costs immediately.")

        if savings < income * 0.20:
            suggestions.append("TIP: Try to save at least 20% of your income each month.")

        if expenses > income * 0.70:
            suggestions.append("ALERT: Your expenses are very high relative to your income.")

        if savings > income * 0.30:
            suggestions.append("GREAT: You have strong savings. Consider investing the surplus.")

        if not suggestions:
            suggestions.append("Your finances look balanced. Keep it up!")

        return suggestions



    def create_widgets(self):
        clear_output()

        title = widgets.Label(value=" AI Finance Management System ")

        data = self.load_data()
        if data is not None and not data.empty:
            record_label = widgets.Label(value=f"Records saved in Google Drive: {len(data)}")
        else:
            record_label = widgets.Label(value="No records saved yet. Start by tracking your expenses.")

        track_btn   = widgets.Button(description="Track Expenses",    button_style='success')
        invest_btn  = widgets.Button(description="Smart Investments",  button_style='info')
        tips_btn    = widgets.Button(description="Money Tips",         button_style='warning')
        health_btn  = widgets.Button(description="Data Health",        button_style='')
        fix_btn     = widgets.Button(description="Auto Fix Data",      button_style='')
        compare_btn = widgets.Button(description="Compare Models",     button_style='')

        track_btn.on_click(self.track_expenses)
        invest_btn.on_click(self.see_smart_investments)
        tips_btn.on_click(self.see_money_tips)
        health_btn.on_click(self.data_health)
        fix_btn.on_click(self.auto_fix)
        compare_btn.on_click(self.compare_models)

        row1 = widgets.HBox([track_btn, invest_btn, tips_btn])
        row2 = widgets.HBox([health_btn, fix_btn, compare_btn])

        display(title, record_label, row1, row2)


    def track_expenses(self, _):
        clear_output()

        # All sections use separate Output widgets so each one
        # is guaranteed to display independently
        title_out      = widgets.Output()
        summary_out    = widgets.Output()
        chart_out      = widgets.Output()
        suggestion_out = widgets.Output()
        back_out       = widgets.Output()

        income_input   = widgets.FloatText(description="Income:",   style={'description_width': 'initial'})
        expenses_input = widgets.FloatText(description="Expenses:", style={'description_width': 'initial'})
        submit_btn     = widgets.Button(description="Submit", button_style='success')

        with title_out:
            display(widgets.Label(value="--- Track Your Expenses ---"))

        def on_submit(_):

            income_val   = income_input.value
            expenses_val = expenses_input.value

            #  Validation
            with summary_out:
                clear_output()
                if income_val <= 0:
                    display(widgets.Label(value="Please enter a valid income greater than 0."))
                    return

            #  Save to Google Drive
            self.last_income   = income_val
            self.last_expenses = expenses_val
            self.save_to_excel(income_val, expenses_val)

            data  = self.load_data()
            total = len(data) if data is not None else 0

            #  Prediction
            linear_model, poly_model, poly = self.train_models()
            if linear_model is not None:
                income_df   = pd.DataFrame([[income_val]], columns=["Income"])
                linear_pred = linear_model.predict(income_df)[0]
                poly_pred   = poly_model.predict(poly.transform(income_df))[0]
                final_pred  = (linear_pred + poly_pred) / 2
                pred_text   = f"Predicted Expenses Next Month: {self.fmt(final_pred)}"
            else:
                remaining = max(0, 5 - total)
                pred_text = f"Enter {remaining} more record(s) to enable expense prediction."

            #  Summary section
            with summary_out:
                clear_output()
                display(widgets.Label(value="=============================="))
                display(widgets.Label(value="Data saved to Google Drive!"))
                display(widgets.Label(value=f"Total records saved: {total}"))
                display(widgets.Label(value="=============================="))
                display(widgets.Label(value=f"Income:   {self.fmt(income_val)}"))
                display(widgets.Label(value=f"Expenses: {self.fmt(expenses_val)}"))
                display(widgets.Label(value=f"Savings:  {self.fmt(income_val - expenses_val)}"))
                display(widgets.Label(value=""))
                display(widgets.Label(value=pred_text))

            #  Pie chart section
            with chart_out:
                clear_output()
                categories = {
                    "Rent":      0.40,
                    "Utilities": 0.20,
                    "Groceries": 0.15,
                    "Transport": 0.10,
                    "Savings":   0.15,
                }
                labels = list(categories.keys())
                sizes  = [expenses_val * v for v in categories.values()]

                plt.figure(figsize=(5, 5))
                plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140)
                plt.title("Estimated Expense Distribution")
                plt.tight_layout()
                plt.show()

            #  AI Suggestions section
            with suggestion_out:
                clear_output()
                display(widgets.Label(value="=============================="))
                display(widgets.Label(value="AI Suggestions:"))
                display(widgets.Label(value="=============================="))

                suggestions = self.generate_ai_suggestions(income_val, expenses_val)
                for i, suggestion in enumerate(suggestions, 1):
                    display(widgets.Label(value=f"{i}. {suggestion}"))

                display(widgets.Label(value=""))
                display(widgets.Label(value=f"Total Suggestions: {len(suggestions)}"))
                display(widgets.Label(value="=============================="))

            #  Back button section
            with back_out:
                clear_output()
                display(self.back_button())

        submit_btn.on_click(on_submit)

        # Display everything in order - each in its own Output widget
        display(
            title_out,
            income_input,
            expenses_input,
            submit_btn,
            summary_out,
            chart_out,
            suggestion_out,
            back_out
        )



    def see_smart_investments(self, _):
        clear_output()

        display(widgets.Label(value="--- Smart Investment Ideas (Global) ---"))
        display(widgets.Label(value=""))
        display(widgets.Label(value="== Stock Markets =="))
        display(widgets.Label(value="1. Index Funds       - Invest in a basket of top companies worldwide (e.g. S&P 500, MSCI World)."))
        display(widgets.Label(value="2. Individual Stocks - Buy shares in companies like Apple, Tesla, Samsung, or local firms."))
        display(widgets.Label(value="3. ETFs              - Exchange-Traded Funds that track markets, sectors, or commodities."))
        display(widgets.Label(value=""))
        display(widgets.Label(value="== Fixed Income =="))
        display(widgets.Label(value="4. Government Bonds  - Lend money to your government for a fixed return. Low risk."))
        display(widgets.Label(value="5. Corporate Bonds   - Lend money to companies for a higher return than govt bonds."))
        display(widgets.Label(value="6. Fixed Deposits    - Lock money in a bank for a set period to earn guaranteed interest."))
        display(widgets.Label(value=""))
        display(widgets.Label(value="== Real Assets =="))
        display(widgets.Label(value="7. Real Estate       - Buy property to earn rental income or sell at a profit later."))
        display(widgets.Label(value="8. REITs             - Real Estate Investment Trusts let you invest in property without buying it."))
        display(widgets.Label(value="9. Gold & Commodities- Physical gold, silver, or oil as a hedge against inflation."))
        display(widgets.Label(value=""))
        display(widgets.Label(value="== Modern & Digital =="))
        display(widgets.Label(value="10. Mutual Funds     - Professionally managed funds that pool money from many investors."))
        display(widgets.Label(value="11. Cryptocurrency   - High risk, high reward digital assets like Bitcoin or Ethereum."))
        display(widgets.Label(value="12. Robo-Advisors    - Apps that automatically invest your money based on your risk level."))
        display(widgets.Label(value=""))
        display(widgets.Label(value="== Community & Local =="))
        display(widgets.Label(value="13. Cooperatives     - Join a local savings group or credit union for loans and dividends."))
        display(widgets.Label(value="14. Small Business   - Start or invest in a local business for long-term income."))
        display(widgets.Label(value=""))
        display(widgets.Label(value="TIP: Always research before investing. Diversify to reduce risk."))
        display(widgets.Label(value="Explore global markets at: https://finance.yahoo.com/"))
        display(self.back_button())



    def see_money_tips(self, _):
        clear_output()
        display(widgets.Label(value="--- Personalised Money Tips ---"))

        if self.last_income is None or self.last_expenses is None:
            display(widgets.Label(value="Please enter your income and expenses first using Track Expenses."))
            display(self.back_button())
            return

        income_val   = self.last_income
        expenses_val = self.last_expenses

        savings_target     = income_val * 0.20
        max_expenses_limit = income_val * 0.50
        investment_amount  = income_val * 0.30

        display(widgets.Label(value="=============================="))
        display(widgets.Label(value=f"Your Income:                    {self.fmt(income_val)}"))
        display(widgets.Label(value=f"Your Expenses:                  {self.fmt(expenses_val)}"))
        display(widgets.Label(value=f"Your Savings:                   {self.fmt(income_val - expenses_val)}"))
        display(widgets.Label(value="=============================="))
        display(widgets.Label(value=f"Savings Target (20%):           {self.fmt(savings_target)}"))
        display(widgets.Label(value=f"Max Recommended Expenses (50%): {self.fmt(max_expenses_limit)}"))
        display(widgets.Label(value=f"Investment Potential (30%):     {self.fmt(investment_amount)}"))
        display(widgets.Label(value=""))

        if expenses_val > income_val:
            display(widgets.Label(value="WARNING: You are overspending. Cut unnecessary costs immediately."))
        elif expenses_val > max_expenses_limit:
            display(widgets.Label(value="ALERT: Your expenses are above the recommended 50% limit."))
        else:
            display(widgets.Label(value="GOOD: Your spending is under control. Well done!"))

        if (income_val - expenses_val) < savings_target:
            display(widgets.Label(value="TIP: You are saving less than 20%. Try to reduce spending."))
        else:
            display(widgets.Label(value="GREAT: You have a strong savings habit. Consider investing more."))

        display(widgets.Label(value=""))
        display(widgets.Label(value="RULE: Follow the 50/30/20 rule - 50% needs, 30% wants, 20% savings."))
        self.speak("Here are your personalised money tips based on your financial data.")
        display(self.back_button())



    def data_health(self, _):
        clear_output()
        display(widgets.Label(value="--- Data Health Check ---"))

        data = self.load_data()
        if data is None or data.empty:
            display(widgets.Label(value="No data found. Please track some expenses first."))
            display(self.back_button())
            return

        total_records  = len(data)
        missing_values = data.isnull().sum().sum()
        duplicates     = data.duplicated().sum()
        invalid_rows   = (data["Expenses"] > data["Income"]).sum()
        avg_income     = data["Income"].mean()
        avg_expenses   = data["Expenses"].mean()
        avg_savings    = avg_income - avg_expenses

        display(widgets.Label(value="=============================="))
        display(widgets.Label(value=f"Total Records:     {total_records}"))
        display(widgets.Label(value=f"Missing Values:    {missing_values}"))
        display(widgets.Label(value=f"Duplicate Rows:    {duplicates}"))
        display(widgets.Label(value=f"Overspending Rows: {invalid_rows}"))
        display(widgets.Label(value="=============================="))
        display(widgets.Label(value=f"Average Income:    {self.fmt(avg_income)}"))
        display(widgets.Label(value=f"Average Expenses:  {self.fmt(avg_expenses)}"))
        display(widgets.Label(value=f"Average Savings:   {self.fmt(avg_savings)}"))
        display(widgets.Label(value="=============================="))

        if missing_values == 0 and invalid_rows == 0 and duplicates == 0:
            display(widgets.Label(value="Status: Data looks healthy!"))
        else:
            display(widgets.Label(value="Status: Issues found. Use Auto Fix Data to clean your data."))

        fig, ax = plt.subplots(figsize=(8, 4))
        x = range(1, total_records + 1)
        ax.bar([i - 0.2 for i in x], data["Income"],   width=0.4, label="Income",   color='steelblue')
        ax.bar([i + 0.2 for i in x], data["Expenses"], width=0.4, label="Expenses", color='tomato')
        ax.set_xlabel("Record Number")
        ax.set_ylabel("Amount")
        ax.set_title("Income vs Expenses - All Records")
        ax.legend()
        plt.tight_layout()
        plt.show()

        display(self.back_button())



    def auto_fix(self, _):
        clear_output()
        display(widgets.Label(value="--- Auto Fix Data ---"))

        data = self.load_data()
        if data is None or data.empty:
            display(widgets.Label(value="No data found. Nothing to fix."))
            display(self.back_button())
            return

        original_count = len(data)
        data = data.dropna()
        data = data.drop_duplicates()
        data = data[(data["Income"] > 0) & (data["Expenses"] > 0)]
        fixed_count = original_count - len(data)
        data.to_excel(self.file_name, index=False)

        display(widgets.Label(value="=============================="))
        display(widgets.Label(value="Auto-fix complete. Cleaned data saved to Google Drive."))
        display(widgets.Label(value="=============================="))
        display(widgets.Label(value=f"Records before cleaning: {original_count}"))
        display(widgets.Label(value=f"Records removed:         {fixed_count}"))
        display(widgets.Label(value=f"Records remaining:       {len(data)}"))
        display(self.back_button())



    def compare_models(self, _):
        clear_output()
        display(widgets.Label(value="--- Compare Prediction Models ---"))

        data = self.load_data()
        if data is None or len(data) < 5:
            display(widgets.Label(value="Not enough data. Please enter at least 5 records first."))
            display(self.back_button())
            return

        X = data[["Income"]]
        y = data["Expenses"].values

        linear_model = LinearRegression()
        linear_model.fit(X, y)
        linear_preds = linear_model.predict(X)
        linear_mse   = mean_squared_error(y, linear_preds)

        poly = PolynomialFeatures(degree=2)
        X_poly = poly.fit_transform(X)
        poly_model = LinearRegression()
        poly_model.fit(X_poly, y)
        poly_preds = poly_model.predict(X_poly)
        poly_mse   = mean_squared_error(y, poly_preds)

        better = "Polynomial" if poly_mse < linear_mse else "Linear"

        display(widgets.Label(value="=============================="))
        display(widgets.Label(value=f"Linear Regression MSE:     {linear_mse:,.2f}"))
        display(widgets.Label(value=f"Polynomial Regression MSE: {poly_mse:,.2f}"))
        display(widgets.Label(value="=============================="))
        display(widgets.Label(value=f"Better Model: {better} Regression (lower MSE = better fit)"))
        display(widgets.Label(value="=============================="))

        X_vals      = data["Income"].values
        sorted_idx  = np.argsort(X_vals)
        X_sorted    = X_vals[sorted_idx]
        y_sorted    = y[sorted_idx]
        X_sorted_df = pd.DataFrame(X_sorted, columns=["Income"])

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.scatter(X_sorted, y_sorted, color='gray', label='Actual Expenses', zorder=3)
        ax.plot(X_sorted, linear_model.predict(X_sorted_df), color='steelblue',
                label=f'Linear (MSE: {linear_mse:,.0f})', linewidth=2)
        ax.plot(X_sorted, poly_model.predict(poly.transform(X_sorted_df)), color='tomato',
                linestyle='--', label=f'Polynomial (MSE: {poly_mse:,.0f})', linewidth=2)
        ax.set_xlabel("Income")
        ax.set_ylabel("Expenses")
        ax.set_title("Linear vs Polynomial Regression")
        ax.legend()
        plt.tight_layout()
        plt.show()

        display(self.back_button())



finance_app = PersonalFinanceApp()

Label(value=' AI Finance Management System ')

Label(value='Records saved in Google Drive: 43')